# Where does our growth come from?

**Kalpa Retail, Week 1 Day 1.** Revenue grew 4 percent last year against a plan of 15. The board
wants a growth plan within a month, and marketing has asked for Rs 12 crore to acquire new
customers.

Meera Raghavan, CEO, to the new data team:

> "Before I sign anything, I want to understand our own sales. What is 'sales' made of? Where does
> revenue come from, by customer type and channel? Is acquisition even the branch that is short?"

Anand Iyer, the finance controller, adds one rule: **no averages, because one business customer
can move an average.**

This notebook answers the first question. It computes what sales is made of, on thirty real Kalpa
orders that nobody has explained to you.

In [1]:
import pathlib
import sys

root = next(p for p in pathlib.Path.cwd().resolve().parents if (p / "scripts" / "c2kit.py").exists())
sys.path.insert(0, str(root / "scripts"))
import c2kit as kit

ORDERS = kit.load_records("C2_W01_D01_orders_STUDENT.py")
print(f"{len(ORDERS)} Kalpa Retail orders loaded")
print(ORDERS[0])

30 Kalpa Retail orders loaded
{'order_id': 'KR-01001', 'customer_id': 'C-0101', 'segment': 'Retail-Core', 'channel': 'app', 'order_date': '2026-07-21', 'amount': 2300, 'status': 'delivered'}


## MAP: the tree before the code

Revenue is not one number. It is a product of five things, and growth comes from moving one of
them at a time. Each branch costs something different to move, which is why naming the branch
matters more than computing the total.

In [2]:
kit.tree(
    {"label": "REVENUE",
     "branches": [
         ("who", {"label": "customers"}),
         ("how often", {"label": "orders per customer"}),
         ("how full", {"label": "items per order"}),
         ("how dear", {"label": "price per item"}),
         ("less", {"label": "discounts"}),
     ]},
    taken=["who", "how often"],
    title="The revenue tree. The two lit branches are the ones this file can answer.")

The two lit branches are the only ones today's file can answer, because it carries orders and
customers and no items. Naming what you cannot compute is half the answer; asking for it is the
other half.

## DO: one order is a dictionary

A record is a set of named boxes. You reach into it by name, never by position, which is why the
order of the keys never matters and a typo in a key is fatal.

In [3]:
first = ORDERS[0]
for key, value in first.items():
    print(f"{key:14s} {value!r}")

order_id       'KR-01001'
customer_id    'C-0101'
segment        'Retail-Core'
channel        'app'
order_date     '2026-07-21'
amount         2300
status         'delivered'


## DO: counting is an accumulator

Start at zero, walk the list, add one each time. Every leaf on the tree is a variation of these
three lines.

In [4]:
count = 0
for order in ORDERS:
    count = count + 1
print("orders:", count)

orders: 30


## SEE: summing is the same shape, until it is not

The only thing that changes is what gets added. Count adds one; sum adds the value.

In [5]:
try:
    total = 0
    for order in ORDERS:
        total = total + order["amount"]
    print("revenue:", total)
except TypeError as e:
    print("TypeError:", e)

TypeError: unsupported operand type(s) for +: 'int' and 'str'


### The break, and how to read it

`unsupported operand type(s) for +=: 'int' and 'str'` is four facts in one line: the kind of
problem, the operation that failed, what sat on each side of it, and where Python gave up.

One amount in this file arrived as the text `"4500"` rather than the number `4500`. Python will
not add a word to a number.

In [6]:
offenders = [(i, o["order_id"], o["amount"]) for i, o in enumerate(ORDERS)
             if isinstance(o["amount"], str)]
kit.table(["row", "order_id", "amount as stored"], offenders,
          caption="Every amount that is text rather than a number")

row,order_id,amount as stored
7,KR-01008,4500


The fix for today is `int()`, and it has a cost: it turns a loud problem into a silent assumption.
Nobody has written down that this file holds text where numbers belong. On Wednesday that
assumption gets a log entry and a reason.

In [7]:
revenue = 0
for order in ORDERS:
    revenue = revenue + int(order["amount"])
print(f"revenue: Rs {revenue:,}")

revenue: Rs 544,810


## DO: the other two leaves

Customers is a count of distinct ids. Orders per customer is the first real rate of the
programme, so it is the first one that needs its denominator said out loud.

In [8]:
customers = set()
for order in ORDERS:
    customers.add(order["customer_id"])

orders_per_customer = len(ORDERS) / len(customers)
print(f"customers: {len(customers)}")
print(f"orders per customer: {orders_per_customer:.2f}  "
      f"({len(ORDERS)} orders divided by {len(customers)} distinct customers)")

customers: 23
orders per customer: 1.30  (30 orders divided by 23 distinct customers)


## CHECK: the leaves, before anything is claimed

In [9]:
kit.check("thirty orders in the file", len(ORDERS) == 30, f"got {len(ORDERS)}")
kit.check("revenue is Rs 5,44,810", revenue == 544810, f"got Rs {revenue:,}")
kit.check("twenty-three distinct customers", len(customers) == 23, f"got {len(customers)}")
kit.check("orders per customer is 1.30", round(orders_per_customer, 2) == 1.30,
          f"got {orders_per_customer:.2f}")
kit.check("exactly one amount is stored as text", len(offenders) == 1,
          f"got {len(offenders)}")

## SEE: which "sales" did we just compute?

Four honest answers live inside one word. Finance recognises delivered revenue. Say which one you
used, every time, or the room argues about the wrong number.

In [10]:
by_status = {}
for order in ORDERS:
    bucket = by_status.setdefault(order["status"], [0, 0])
    bucket[0] += 1
    bucket[1] += int(order["amount"])

kit.table(["status", "orders", "revenue"],
          [(s, n, f"Rs {v:,}") for s, (n, v) in sorted(by_status.items())],
          caption="The same thirty orders, split by what happened to them")

status,orders,revenue
cancelled,4,"Rs 9,050"
delivered,21,"Rs 520,790"
returned,5,"Rs 14,970"


## SEE: the average order, and the average that lies

The mean is the total divided by the count. The median is the value in the middle once the
amounts are sorted. On a well-behaved file they sit close together, and the gap between them is
itself a finding.

In [11]:
amounts = sorted(int(o["amount"]) for o in ORDERS)
mean = sum(amounts) / len(amounts)
middle = len(amounts) // 2
median = amounts[middle] if len(amounts) % 2 else (amounts[middle - 1] + amounts[middle]) / 2

print(f"mean order value:   Rs {mean:,.0f}")
print(f"median order value: Rs {median:,.0f}")
print(f"the mean is {mean / median:.0f} times the median")

mean order value:   Rs 18,160
median order value: Rs 2,205
the mean is 8 times the median


### Where the gap comes from

Sort the amounts and look at the top of the list.

In [12]:
kit.table(["rank", "amount"],
          [(i + 1, f"Rs {a:,}") for i, a in enumerate(reversed(amounts[-5:]))],
          caption="The five largest orders in the file")

rank,amount
1,"Rs 480,000"
2,"Rs 4,500"
3,"Rs 4,100"
4,"Rs 4,090"
5,"Rs 3,830"


In [13]:
largest = amounts[-1]
without = amounts[:-1]
mean_without = sum(without) / len(without)
share = 100 * largest / sum(amounts)

print(f"largest order:              Rs {largest:,}")
print(f"its share of all revenue:   {share:.0f} percent")
print(f"mean without it:            Rs {mean_without:,.0f}")
print(f"median (unchanged):         Rs {median:,.0f}")

largest order:              Rs 480,000
its share of all revenue:   88 percent
mean without it:            Rs 2,235
median (unchanged):         Rs 2,205


One order carries most of the revenue. Take it out and the mean lands next door to the median.
That is exactly what the finance controller meant by "one business customer can move an average",
and it is why a typical order is described by the median and never by the mean.

In [14]:
kit.flow(["30 orders", "sort by amount", "one order is 88 percent", "report the median"],
         lit=2, title="Why the median is the honest description of a typical order")

## CHECK: the average that lies

In [15]:
kit.check("the mean sits far above the median", mean > 5 * median,
          f"mean Rs {mean:,.0f} against median Rs {median:,.0f}")
kit.check("one order carries most of the revenue", share > 80, f"{share:.0f} percent")
kit.check("removing it brings the mean to the median",
          abs(mean_without - median) < 0.25 * median,
          f"Rs {mean_without:,.0f} against Rs {median:,.0f}")
kit.check("the median did not move", median == 2205, f"Rs {median:,.0f}")

## SUM: the tree with today's numbers on it

In [16]:
kit.tree(
    {"label": "REVENUE Rs 5,44,810",
     "branches": [
         ("23", {"label": "customers"}),
         ("1.30", {"label": "orders per customer"}),
         ("no items", {"label": "items per order"}),
         ("no items", {"label": "price per item"}),
         ("no field", {"label": "discounts"}),
     ]},
    taken=["23", "1.30"],
    title="Two branches answered, three named as unanswerable and asked for")

In [17]:
kit.check_summary()

## The sentence you would send Meera

> "Across thirty orders the typical basket is about Rs 2,200, and one corporate order carries 88
> percent of the revenue, so the average order value of Rs 18,160 describes nothing in the file.
> Before we fund acquisition I want to check whether the branch that moved is customers or how
> often they come back. I will have that by Thursday."

Claim, caveat, next step. That shape returns every day this week.

---

**What this file cannot tell you.** It has no items, so three of the five branches are unanswered.
It has no second period, so nothing here says anything moved. Tomorrow's file has two quarters,
and the question becomes which branch moved rather than what the branches are.